In [1]:
import requests
import pandas as pd
import textstat

In [2]:
file_path = "filtered_dataset.csv"  # change if needed
df = pd.read_csv(file_path)

df.head()

,question,answer
0,How do muscles grow?,"I hope this answer qualifies as technical, yet..."
1,Why is chickenpox worse as an adult?,It's mostly due to the difference in immune sy...
2,"Why are some fish bones edible, and others are...",They are small and soft so it does not matter ...
3,Why has the Mars Rover Opportunity's Lithium I...,NASA requirements lean toward the 'overenginee...
4,If the inside of my microwave is made of metal...,The metal interior of the oven is grounded. It...


In [3]:
def compute_readability(text):
    fre = textstat.flesch_reading_ease(text)
    fkgl = textstat.flesch_kincaid_grade(text)
    return fre, fkgl

df["FRE"], df["FKGL"] = zip(*df["answer"].apply(compute_readability))

In [4]:
df.to_csv("filtered_dataset_readability.csv", index=False)

In [5]:
df[["question", "FRE", "FKGL"]].head(10)

,question,FRE,FKGL
0,How do muscles grow?,61.818947,10.705564
1,Why is chickenpox worse as an adult?,32.379826,14.498237
2,"Why are some fish bones edible, and others are...",82.296667,8.488148
3,Why has the Mars Rover Opportunity's Lithium I...,56.462500,10.352500
4,If the inside of my microwave is made of metal...,79.273736,7.005172
5,How does bugspray kills bugs?,44.815175,10.155263
6,Why are the things that taste the best bad for...,67.857632,7.840376
7,How do devices know the amount of charge left ...,77.857857,4.387143
8,Why are my muscles sore after jumping in cold ...,37.383333,10.695000
9,Why doesn't it rain salt water?,68.777391,7.747578


In [11]:
beginner_template = "Explain this like I'm 5 using very simple words and short sentences:\n{}"

intermediate_template = "Explain this clearly in a simple but slightly detailed way, avoiding too much technical jargon:\n{}"

expert_template = "Give a detailed and technical explanation using precise terminology and concepts:\n{}"

In [12]:
df["beginner_prompt"] = df["question"].apply(lambda q: beginner_template.format(q))
df["intermediate_prompt"] = df["question"].apply(lambda q: intermediate_template.format(q))
df["expert_prompt"] = df["question"].apply(lambda q: expert_template.format(q))

df[["question", "beginner_prompt", "intermediate_prompt", "expert_prompt"]].head()

,question,beginner_prompt,intermediate_prompt,expert_prompt
0,How do muscles grow?,Explain this like I'm 5 using very simple word...,Explain this clearly in a simple but slightly ...,Give a detailed and technical explanation usin...
1,Why is chickenpox worse as an adult?,Explain this like I'm 5 using very simple word...,Explain this clearly in a simple but slightly ...,Give a detailed and technical explanation usin...
2,"Why are some fish bones edible, and others are...",Explain this like I'm 5 using very simple word...,Explain this clearly in a simple but slightly ...,Give a detailed and technical explanation usin...
3,Why has the Mars Rover Opportunity's Lithium I...,Explain this like I'm 5 using very simple word...,Explain this clearly in a simple but slightly ...,Give a detailed and technical explanation usin...
4,If the inside of my microwave is made of metal...,Explain this like I'm 5 using very simple word...,Explain this clearly in a simple but slightly ...,Give a detailed and technical explanation usin...


In [13]:
df["beginner_output"] = df["beginner_prompt"].apply(generate_ollama)
df["intermediate_output"] = df["intermediate_prompt"].apply(generate_ollama)
df["expert_output"] = df["expert_prompt"].apply(generate_ollama)

In [14]:
print(df[["question", "beginner_output"]].head())

                                            question  \
0                               How do muscles grow?   
1               Why is chickenpox worse as an adult?   
2  Why are some fish bones edible, and others are...   
3  Why has the Mars Rover Opportunity's Lithium I...   
4  If the inside of my microwave is made of metal...   

                                     beginner_output  
0  Your muscles get stronger when you use them! W...  
1  Chickenpox is like a bad cold for grown-ups.  ...  
2  Some fish have tiny bones inside them.  It hel...  
3  Imagine a tiny house with lots of toys inside....  
4  Imagine your microwave is a little oven. It ha...  


In [17]:
df["beginner_FRE"], df["beginner_FKGL"] = zip(*df["beginner_output"].apply(compute_readability))
df["intermediate_FRE"], df["intermediate_FKGL"] = zip(*df["intermediate_output"].apply(compute_readability))
df["expert_FRE"], df["expert_FKGL"] = zip(*df["expert_output"].apply(compute_readability))

In [18]:
cols_to_save = [
    "question",
    "beginner_output",
    "intermediate_output",
    "expert_output",
    "beginner_FRE", "beginner_FKGL",
    "intermediate_FRE", "intermediate_FKGL",
    "expert_FRE", "expert_FKGL"
]

df[cols_to_save].to_csv("dataset_3level.csv", index=False)

df.to_json("dataset_3level.json", orient="records", indent=2)

In [19]:
# Beginner (Easy)
B_FRE_MIN = 70
B_FKGL_MAX = 6

# Intermediate
I_FRE_MIN = 40
I_FRE_MAX = 70
I_FKGL_MIN = 6
I_FKGL_MAX = 10

# Expert (Hard)
E_FRE_MAX = 50
E_FKGL_MIN = 10

In [20]:
df["b_valid"] = (
    (df["beginner_FRE"] >= B_FRE_MIN) &
    (df["beginner_FKGL"] <= B_FKGL_MAX)
)

df["i_valid"] = (
    (df["intermediate_FRE"] >= I_FRE_MIN) &
    (df["intermediate_FRE"] <= I_FRE_MAX) &
    (df["intermediate_FKGL"] >= I_FKGL_MIN) &
    (df["intermediate_FKGL"] <= I_FKGL_MAX)
)

df["e_valid"] = (
    (df["expert_FRE"] <= E_FRE_MAX) &
    (df["expert_FKGL"] >= E_FKGL_MIN)
)

In [21]:
df_clean = df[
    df["b_valid"] &
    df["i_valid"] &
    df["e_valid"]
].copy()

print("Total clean samples:", len(df_clean))

Total clean samples: 26


In [22]:
cols_to_save = [
    "question",
    "beginner_output",
    "intermediate_output",
    "expert_output",
    "beginner_FRE", "beginner_FKGL",
    "intermediate_FRE", "intermediate_FKGL",
    "expert_FRE", "expert_FKGL"
]

df_clean[cols_to_save].to_csv("filtered_dataset_3level.csv", index=False)

df_clean.to_json("filtered_dataset_3level.json", orient="records", indent=2)

In [23]:
import numpy as np

def cohens_d(x, y):
    x = np.array(x)
    y = np.array(y)
    
    mean_x = np.mean(x)
    mean_y = np.mean(y)
    
    std_x = np.std(x, ddof=1)
    std_y = np.std(y, ddof=1)
    
    pooled_std = np.sqrt((std_x**2 + std_y**2) / 2)
    
    return (mean_x - mean_y) / pooled_std


# FRE → Beginner should be higher
d_fre = cohens_d(df_clean["beginner_FRE"], df_clean["expert_FRE"])

# FKGL → Expert should be higher, so flip
d_fkgl = cohens_d(df_clean["expert_FKGL"], df_clean["beginner_FKGL"])

print("Cohen's d (FRE):", d_fre)
print("Cohen's d (FKGL):", d_fkgl)

Cohen's d (FRE): 5.819736973736832
Cohen's d (FKGL): 5.836502868784998


In [24]:
import json

steering_pairs = []

for _, row in df_clean.iterrows():
    steering_pairs.append({
        "question": row["question"],
        "easy": row["beginner_output"],
        "hard": row["expert_output"]
    })

print("Total steering pairs:", len(steering_pairs))

Total steering pairs: 26


In [25]:
with open("steering_pairs.json", "w") as f:
    json.dump(steering_pairs, f, indent=2)

print("Saved steering_pairs.json")

pairs_df = pd.DataFrame(steering_pairs)
pairs_df.to_csv("steering_pairs.csv", index=False)

Saved steering_pairs.json
